In [0]:
%sql
CREATE TABLE IF NOT EXISTS pipeline_log
(
    run_id STRING,
    pipeline_name STRING,
    task_name STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    records_processed BIGINT,
    error_message STRING
)
USING DELTA;

In [0]:
%sql
SHOW TABLES;

In [0]:
from uuid import uuid4

run_id = str(uuid4())

print("Run ID :", run_id)

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("task_name", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("records_processed", LongType(), True),
    StructField("error_message", StringType(), True)
])

log_df = spark.createDataFrame([
    (run_id, "Ecommerce_ETL_Pipeline", "Bronze_Customers", datetime.now(), None, "STARTED", 0, None)
], schema)

log_df.write.mode("append").saveAsTable("pipeline_log")

print("Pipeline START log inserted successfully.")

In [0]:
%sql
SELECT *
FROM pipeline_log;

In [0]:
%sql
UPDATE pipeline_log
SET
    status = 'SUCCESS',
    end_time = current_timestamp(),
    records_processed = (
        SELECT COUNT(*)
        FROM bronze_customers
    )
WHERE
    run_id = (
        SELECT MAX(run_id)
        FROM pipeline_log
    )
AND task_name = 'Bronze_Customers';

In [0]:
%sql
SELECT *
FROM pipeline_log;

In [0]:
%sql
INSERT INTO pipeline_log
VALUES
(
'TEST_RUN_001',
'Ecommerce_ETL_Pipeline',
'Silver_Customers',
current_timestamp(),
current_timestamp(),
'FAILED',
0,
'Schema Validation Failed'
);

In [0]:
%sql
SELECT
    pipeline_name,
    task_name,
    status,
    records_processed,
    start_time,
    end_time,
    error_message
FROM pipeline_log
ORDER BY start_time DESC;